In [1]:
from torch_geometric.nn import HGTConv, Linear
from torch_geometric.loader import HGTLoader
from torch_geometric.data import HeteroData
import torch.nn.functional as F
# import pickle5 as pickle
import pickle
import torch.nn as nn
import pandas as pd
from utils import *
import random
import torch
import copy

from tqdm import tqdm

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
node_type1 = 'drug'
node_type2 = 'disease'
rel = 'indication'

In [3]:
config = {
    "num_samples": 512,
    "batch_size": 164,
    "dropout": 0.5,
    "epochs": 300
}

# Load data

In [4]:
primekg_file = '../data/kg.csv'
df = pd.read_csv(primekg_file, sep =",")

/tmp/ipykernel_56381/259066371.py:2: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(primekg_file, sep =",")


In [5]:
# 查看 DataFrame 基本信息（结构）
print("=== DataFrame Info (结构) ===")
print(df.info())

# 查看前 5 行数据（快速预览内容）
print("\n=== First 5 rows (前5行数据) ===")
print(df.head())

# 可选：查看各列的唯一值数量（辅助理解）
print("\n=== Number of unique values per column ===")
print(df.nunique())

=== DataFrame Info (结构) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8100498 entries, 0 to 8100497
Data columns (total 12 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   relation          object
 1   display_relation  object
 2   x_index           int64 
 3   x_id              object
 4   x_type            object
 5   x_name            object
 6   x_source          object
 7   y_index           int64 
 8   y_id              object
 9   y_type            object
 10  y_name            object
 11  y_source          object
dtypes: int64(2), object(10)
memory usage: 741.6+ MB
None

=== First 5 rows (前5行数据) ===
          relation display_relation  x_index  x_id        x_type  x_name  \
0  protein_protein              ppi        0  9796  gene/protein  PHYHIP   
1  protein_protein              ppi        1  7918  gene/protein  GPANK1   
2  protein_protein              ppi        2  8233  gene/protein   ZRSR2   
3  protein_protein              ppi        3  4

In [6]:
drug_disease_pairs = df[df['relation']==rel]
drugs, diseases = [], []

for i, row in drug_disease_pairs.iterrows():
    if row['x_type'] == node_type1:
        drugs.append(row['x_index'])
    if row['x_type'] == node_type2:
        diseases.append(row['x_index'])
    
    if row['y_type'] == node_type1:
        drugs.append(row['y_index'])
    if row['y_type'] == node_type2:
        diseases.append(row['y_index'])
        
drugs, diseases = list(set(drugs)), list(set(diseases))

In [7]:
# ========== 插入点：统计 BEFORE ==========
def count_nodes(df):
    nodes_x = df[['x_type', 'x_index']].rename(columns={'x_type': 'type', 'x_index': 'id'})
    nodes_y = df[['y_type', 'y_index']].rename(columns={'y_type': 'type', 'y_index': 'id'})
    all_nodes = pd.concat([nodes_x, nodes_y], ignore_index=True)
    return all_nodes.drop_duplicates().groupby('type').size().to_dict()

# 移除前统计（此时 df 尚未被修改！）
node_counts_before = count_nodes(df)
edge_counts_before = df['relation'].value_counts().to_dict()
# =======================================

In [8]:
to_remove = df[df['x_type']==node_type1]
to_remove = to_remove[~to_remove['x_index'].isin(drugs)]
df.drop(to_remove.index, inplace = True)

In [9]:
to_remove = df[df['y_type']==node_type1]
to_remove = to_remove[~to_remove['y_index'].isin(drugs)]
df.drop(to_remove.index, inplace = True)

In [10]:
to_remove = df[df['x_type']==node_type2]
to_remove = to_remove[~to_remove['x_index'].isin(diseases)]
df.drop(to_remove.index, inplace = True)

In [11]:
to_remove = df[df['y_type']==node_type2]
to_remove = to_remove[~to_remove['y_index'].isin(diseases)]
df.drop(to_remove.index, inplace = True)

In [13]:
# ========== 统计 AFTER 并保存 ==========
node_counts_after = count_nodes(df)
edge_counts_after = df['relation'].value_counts().to_dict()

# ---------- 构建 Table 2: Nodes ----------
all_node_types = sorted(set(node_counts_before.keys()) | set(node_counts_after.keys()))
table2_rows = [
    {
        'Entity': typ,
        'Count before removal': node_counts_before.get(typ, 0),
        'Count after removal': node_counts_after.get(typ, 0),
        'Removal percent': round(
            ((node_counts_before.get(typ, 0) - node_counts_after.get(typ, 0)) 
             / node_counts_before.get(typ, 0) * 100) if node_counts_before.get(typ, 0) > 0 else 0.0,
            2
        )
    }
    for typ in all_node_types
]

# 计算 Total 行（Nodes）
total_before_nodes = sum(row['Count before removal'] for row in table2_rows)
total_after_nodes = sum(row['Count after removal'] for row in table2_rows)
total_percent_nodes = round(
    ((total_before_nodes - total_after_nodes) / total_before_nodes * 100) if total_before_nodes > 0 else 0.0,
    2
)

# 添加 Total 行
table2_rows.append({
    'Entity': 'Total',
    'Count before removal': total_before_nodes,
    'Count after removal': total_after_nodes,
    'Removal percent': total_percent_nodes
})

table2 = pd.DataFrame(table2_rows)

# ---------- 构建 Table 3: Edges ----------
all_relations = sorted(set(edge_counts_before.keys()) | set(edge_counts_after.keys()))
table3_rows = [
    {
        'Entity': rel,
        'Count before removal': edge_counts_before.get(rel, 0),
        'Count after removal': edge_counts_after.get(rel, 0),
        'Removal percent': round(
            ((edge_counts_before.get(rel, 0) - edge_counts_after.get(rel, 0)) 
             / edge_counts_before.get(rel, 0) * 100) if edge_counts_before.get(rel, 0) > 0 else 0.0,
            2
        )
    }
    for rel in all_relations
]

# 计算 Total 行（Edges）
total_before_edges = sum(row['Count before removal'] for row in table3_rows)
total_after_edges = sum(row['Count after removal'] for row in table3_rows)
total_percent_edges = round(
    ((total_before_edges - total_after_edges) / total_before_edges * 100) if total_before_edges > 0 else 0.0,
    2
)

# 添加 Total 行
table3_rows.append({
    'Entity': 'Total',
    'Count before removal': total_before_edges,
    'Count after removal': total_after_edges,
    'Removal percent': total_percent_edges
})

table3 = pd.DataFrame(table3_rows)

# 保存
table2.to_csv('table2_primekg_node_counts.csv', index=False)
table3.to_csv('table3_primekg_edge_counts.csv', index=False)

print("✅ Tables saved with 'Total' rows!")
# =======================================

✅ Tables saved with 'Total' rows!
